## Stage 03a — WYO Instrument Alignment

Cross-correlate WYO platform instruments against Picarro CH4 (trusted reference).
Writes lag-shifted Parquet to `03_instrument_aligned/`.

> MML instruments (Ultra 321 + Pico 017 MML dates, LGR, Anem, GPS) are handled
> in `03b_align_mml.ipynb`.

| Section | Instrument | Reference | Dates |
|---|---|---|---|
| A | WYO_aerisultra460 | Picarro CH4 | Feb 3–12 |
| B | LANL_aerisultra321 (WYO dates) | Picarro CH4 | Feb 3–12 |
| C | LANL_aerispico017 (WYO dates) | Picarro CH4 | Feb 5–12 |

**Outputs:** `lag_offsets_wyo.json`, `apply_manifest_wyo.json`,
aligned Parquet in `03_instrument_aligned/`.

**Workflow:** Run Imports → Config → Helpers → Load Picarro, then for each section
run Auto-correlate then Widget. After all sections: Save → Apply → Pass-throughs.

In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, REPO_ROOT
from config import MML_DATE_TAGS

display(HTML('<style>.plotly-graph-div { width: 100% !important; }</style>'))
print('Imports OK')

In [ ]:
CH4_COL   = 'CH4_ppm'
MAX_LAG_S = 600

PICARRO_DIR  = STAGE_02_DIR / 'WYO_picarro'
ULTRA460_DIR = STAGE_02_DIR / 'WYO_aerisultra460' / 'Raw'
ULTRA321_DIR = STAGE_02_DIR / 'LANL_aerisultra321' / 'Raw'
PICO017_DIR  = STAGE_02_DIR / 'LANL_aerispico017'  / 'Raw'

print(f'MML_DATE_TAGS ({len(MML_DATE_TAGS)} dates): {sorted(MML_DATE_TAGS)}')
print('Config OK')

In [ ]:
from src.align import resample_series, cross_correlate

def load_parquet_col(path, col):
    return pd.read_parquet(path, columns=[col])[col].dropna()

def load_all_picarro(picarro_dir, col='CH4_ppm'):
    files    = sorted(picarro_dir.glob('*.parquet'))
    combined = pd.concat([load_parquet_col(f, col) for f in files]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined)

def auto_correlate(test_files, ref_data, col='CH4_ppm'):
    suggestions = {}
    print(f"{'IDX':>4}  {'FILE':<55}  {'AUTO LAG':>10}")
    print('-' * 75)
    for i, f in enumerate(test_files):
        sig = resample_series(load_parquet_col(f, col))
        lag = cross_correlate(ref_data, sig)
        suggestions[f.stem] = lag
        print(f'[{i:>2}]  {f.name:<55}  {lag:>+10.0f}s')
    return suggestions

WYO_ONLY_STEMS = {'Ultra100460'}

def is_mml(path):
    stem   = Path(path).stem
    prefix = stem.split('_')[0]
    if prefix in WYO_ONLY_STEMS:
        return False
    date_tag = stem.split('_')[1] if stem.count('_') >= 1 else ''
    return date_tag in MML_DATE_TAGS

def _date_tag(path):
    parts = Path(path).stem.split('_')
    return parts[1] if len(parts) >= 2 else ''

def _git_info():
    try:
        h = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=str(REPO_ROOT), text=True
        ).strip()
        dirty = subprocess.call(['git', 'diff', '--quiet'], cwd=str(REPO_ROOT)) != 0
        return h, dirty
    except Exception:
        return 'unknown', False

def save_lag_offsets_wyo():
    STAGE_03_DIR.mkdir(parents=True, exist_ok=True)
    g = globals()
    def _lags(conf, rej):
        return {k: v for k, v in conf.items() if k not in rej}
    git_hash, git_dirty = _git_info()
    manifest = {
        'stage':     '03a_align_wyo',
        'run_utc':   datetime.now(timezone.utc).isoformat(),
        'git_hash':  git_hash,
        'git_dirty': git_dirty,
        'lags': {
            'WYO_aerisultra460':  _lags(g.get('u460_confirmed',     {}), g.get('u460_rejected',     set())),
            'LANL_aerisultra321': _lags(g.get('u321_wyo_confirmed', {}), g.get('u321_wyo_rejected', set())),
            'LANL_aerispico017':  _lags(g.get('pico_wyo_confirmed', {}), g.get('pico_wyo_rejected', set())),
        },
        'rejected': {
            'WYO_aerisultra460':  sorted(g.get('u460_rejected',     set())),
            'LANL_aerisultra321': sorted(g.get('u321_wyo_rejected', set())),
            'LANL_aerispico017':  sorted(g.get('pico_wyo_rejected', set())),
        },
    }
    with open(STAGE_03_DIR / 'lag_offsets_wyo.json', 'w') as fh:
        json.dump(manifest, fh, indent=2)

print('Helpers loaded.')

In [ ]:
def make_review_widget(ref_data, test_files, test_name, suggestions, confirmed, rejected,
                       ref_name='Ref', save_fn=None, test_cols=None, normalize=False):
    """
    Interactive review widget.

    ref_data : pd.Series | dict[str, pd.Series]
        If dict, each key becomes a separate gray reference trace.
    test_cols : list[str] | None
        Columns to load from each test file.  Default: ['CH4_ppm'].
    normalize : bool
        Z-score each trace for cross-unit overlay.
    """
    if not test_files:
        print(f'{test_name}: no files to review')
        return

    state      = {'idx': 0}
    _ref_dict  = ref_data if isinstance(ref_data, dict) else {ref_name: ref_data}
    _n_ref     = len(_ref_dict)
    _test_cols = test_cols or ['CH4_ppm']
    _n_test    = len(_test_cols)

    REF_COLORS  = ['#888888', '#AAAAAA', '#666666', '#BBBBBB']
    TEST_COLORS = ['#E67E22', '#2980B9', '#27AE60', '#8E44AD']

    fig = go.FigureWidget(layout=go.Layout(
        autosize=True, height=400,
        margin=dict(l=55, r=15, t=62, b=40),
        yaxis=dict(title='z-score' if normalize else _test_cols[0]),
        legend=dict(x=1, y=1, xanchor='right', font=dict(size=10)),
        hovermode='x unified',
    ))
    for i, rk in enumerate(_ref_dict.keys()):
        fig.add_scatter(
            name=rk,
            line=dict(color=REF_COLORS[i % len(REF_COLORS)], width=1.5, dash='dot'),
            opacity=0.85,
        )
    for i, col in enumerate(_test_cols):
        fig.add_scatter(
            name=f'{test_name} -- {col}',
            line=dict(color=TEST_COLORS[i % len(TEST_COLORS)], width=2.5),
        )

    lag_slider = widgets.FloatSlider(
        value=0.0, min=-MAX_LAG_S, max=MAX_LAG_S, step=0.1,
        description='Lag (s):', continuous_update=True, readout_format='.1f',
        layout=widgets.Layout(width='100%'),
        style={'description_width': '60px'},
    )

    btn_prev   = widgets.Button(description='<- Prev',         layout=widgets.Layout(width='85px'))
    btn_next   = widgets.Button(description='Next ->',         layout=widgets.Layout(width='85px'))
    btn_commit = widgets.Button(description='Commit & Next',   button_style='success', layout=widgets.Layout(width='140px'))
    btn_bad    = widgets.Button(description='Mark Bad & Next', button_style='danger',  layout=widgets.Layout(width='150px'))
    log = widgets.Output(layout=widgets.Layout(
        height='90px', overflow_y='auto', border='1px solid #ddd', padding='4px',
    ))

    def _zscore(s):
        mu, sig = s.mean(), s.std()
        return (s - mu) / sig if sig > 0 else s * 0.0

    def _update_fig(idx, lag_s):
        if idx >= len(test_files):
            with fig.batch_update():
                for trace in fig.data:
                    trace.x = []; trace.y = []
            fig.layout.title = dict(
                text=f'{test_name} -- complete  ({len(confirmed)} confirmed, {len(rejected)} rejected)',
                y=0.97, yanchor='top',
            )
            return
        f        = test_files[idx]
        key      = f.stem
        auto_lag = suggestions.get(key, 0.0)
        try:
            test_dict = {
                col: resample_series(load_parquet_col(f, col))
                for col in _test_cols
            }
        except Exception as e:
            fig.layout.title = dict(text=f'ERROR loading {f.name}: {e}')
            return
        first_sig = next(iter(test_dict.values()))
        t0 = first_sig.index[0]  - pd.Timedelta(hours=1)
        t1 = first_sig.index[-1] + pd.Timedelta(hours=1)
        with fig.batch_update():
            for i, (rk, rseries) in enumerate(_ref_dict.items()):
                ref_win  = rseries[t0:t1]
                ref_plot = _zscore(ref_win) if normalize else ref_win
                fig.data[i].x = ref_plot.index
                fig.data[i].y = ref_plot.values
            for i, (col, sig) in enumerate(test_dict.items()):
                shifted_idx = sig.index + pd.Timedelta(seconds=lag_s)
                plot_sig    = _zscore(sig) if normalize else sig
                fig.data[_n_ref + i].x = shifted_idx
                fig.data[_n_ref + i].y = plot_sig.values
                fig.data[_n_ref + i].name = f'{col} (lag={lag_s:+.1f}s)'
        status    = f'  v {confirmed[key]:+.1f}s' if key in confirmed else ('  x bad' if key in rejected else '')
        first_ref = next(iter(_ref_dict.values()))
        n_ref_pts = int(first_ref[t0:t1].notna().sum())
        subtitle  = (f'{len(first_sig):,} rows  {first_sig.index[0].strftime("%H:%M")}'
                     f'-{first_sig.index[-1].strftime("%H:%M")} UTC  ref: {n_ref_pts:,} pts')
        fig.layout.title = dict(
            text=(f'[{idx+1}/{len(test_files)}]  {f.name}'
                  f'  auto={auto_lag:+.1f}s{status}<br><sup>{subtitle}</sup>'),
            y=0.97, yanchor='top',
        )

    def go_to(idx):
        if 0 <= idx < len(test_files):
            lag_slider.value = suggestions.get(test_files[idx].stem, 0.0)
        _update_fig(idx, lag_slider.value)

    lag_slider.observe(lambda change: _update_fig(state['idx'], change['new']), names='value')

    def on_prev(_):
        state['idx'] = max(0, state['idx'] - 1); go_to(state['idx'])
    def on_next(_):
        state['idx'] += 1; go_to(state['idx'])
    def on_commit(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem; lag = round(lag_slider.value, 1)
        confirmed[key] = lag; rejected.discard(key)
        if save_fn: save_fn()
        with log: print(f'COMMITTED  {key}  {lag:+.1f}s')
        state['idx'] += 1; go_to(state['idx'])
    def on_bad(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem
        rejected.add(key); confirmed.pop(key, None)
        if save_fn: save_fn()
        with log: print(f'REJECTED   {key}')
        state['idx'] += 1; go_to(state['idx'])

    btn_prev.on_click(on_prev); btn_next.on_click(on_next)
    btn_commit.on_click(on_commit); btn_bad.on_click(on_bad)
    btn_row = widgets.HBox([btn_prev, btn_bad, btn_commit, btn_next],
                           layout=widgets.Layout(gap='6px', margin='4px 0'))
    display(widgets.VBox([fig, lag_slider, btn_row, log],
                         layout=widgets.Layout(width='100%')))
    go_to(0)

print('Widget helper loaded.')

In [ ]:
print('Loading Picarro reference (may take ~30s)...')
picarro_ref = load_all_picarro(PICARRO_DIR)
print(f'Picarro CH4_ppm: {len(picarro_ref):,} samples')
print(f'  Range: {picarro_ref.index[0]}  ->  {picarro_ref.index[-1]}')

---
## A — Ultra 460 vs Picarro

Ultra 460 is WYO-only (Feb 3–12). All files should have good Picarro overlap.

In [ ]:
u460_files = sorted(ULTRA460_DIR.glob('*.parquet'))
print(f'Ultra 460: {len(u460_files)} files\n')
u460_suggestions = auto_correlate(u460_files, picarro_ref)

In [ ]:
if 'u460_confirmed' not in dir(): u460_confirmed = {}
if 'u460_rejected'  not in dir(): u460_rejected  = set()
make_review_widget(picarro_ref, u460_files, 'Ultra460',
                   u460_suggestions, u460_confirmed, u460_rejected,
                   ref_name='Picarro (ref)', save_fn=save_lag_offsets_wyo)

---
## B — Ultra 321 WYO dates vs Picarro

Ultra 321 WYO dates (Feb 3, 5–12): Picarro overlap, auto-correlate finds lag.
MML files are skipped here — they are aligned in `03b_align_mml.ipynb`.

In [ ]:
u321_all = sorted(ULTRA321_DIR.glob('*.parquet'))
u321_wyo = [f for f in u321_all if not is_mml(f)]
print(f'Ultra 321 total: {len(u321_all)} files')
print(f'Ultra 321 WYO-date files: {len(u321_wyo)}  (MML skipped)\n')
u321_wyo_suggestions = auto_correlate(u321_wyo, picarro_ref)

In [ ]:
if 'u321_wyo_confirmed' not in dir(): u321_wyo_confirmed = {}
if 'u321_wyo_rejected'  not in dir(): u321_wyo_rejected  = set()
make_review_widget(picarro_ref, u321_wyo, 'Ultra321-WYO',
                   u321_wyo_suggestions, u321_wyo_confirmed, u321_wyo_rejected,
                   ref_name='Picarro (ref)', save_fn=save_lag_offsets_wyo)

---
## C — Pico 017 WYO dates vs Picarro

Pico 017 WYO dates (Feb 5–12). MML dates handled in `03b_align_mml.ipynb`.

In [ ]:
pico_all = sorted(PICO017_DIR.glob('*.parquet'))
pico_wyo = [f for f in pico_all if not is_mml(f)]
print(f'Pico 017 total: {len(pico_all)} files')
print(f'Pico 017 WYO-date files: {len(pico_wyo)}  (MML skipped)\n')
pico_wyo_suggestions = auto_correlate(pico_wyo, picarro_ref)

In [ ]:
if 'pico_wyo_confirmed' not in dir(): pico_wyo_confirmed = {}
if 'pico_wyo_rejected'  not in dir(): pico_wyo_rejected  = set()
make_review_widget(picarro_ref, pico_wyo, 'Pico017-WYO',
                   pico_wyo_suggestions, pico_wyo_confirmed, pico_wyo_rejected,
                   ref_name='Picarro (ref)', save_fn=save_lag_offsets_wyo)

---
## Save lag_offsets_wyo.json

Run once all widget sections are done. `lag_offsets_wyo.json` is also auto-saved
after every Commit/Mark Bad click so no work is lost if the kernel dies.

In [ ]:
save_lag_offsets_wyo()
lag_path = STAGE_03_DIR / 'lag_offsets_wyo.json'
print(f'Saved -> {lag_path}\n')
with open(lag_path) as fh:
    saved = json.load(fh)
for inst, lags in saved['lags'].items():
    rej = saved['rejected'].get(inst, [])
    if lags or rej:
        print(f'{inst}: {len(lags)} confirmed, {len(rej)} rejected')
        for stem, lag in sorted(lags.items()):
            print(f'  {stem:<55}  {lag:>+6.0f}s')

---
## Apply lags → `03_instrument_aligned/`

Reads `lag_offsets_wyo.json`.  Every file is routed to one of:
- `{subdir}/*.parquet` — lag-shifted (confirmed lag)
- `{subdir}/bad/*.parquet` — marked bad in widget, copied as-is
- `[WARN]` + 0s lag applied — file reviewed with Next (no commit)

Ultra 321 and Pico 017: `wyo_only=True` skips MML-date files (handled in 03b).

In [ ]:
import shutil
from src.align import raw_stem, apply_lag_to_parquet

def apply_instrument(instrument, subdirs, lags, rejected_stems,
                     apply_spectra=True, wyo_only=False):
    src_inst = STAGE_02_DIR / instrument
    dst_inst = STAGE_03_DIR / instrument
    n_ok = n_bad = n_warn = n_skip = 0
    for subdir in subdirs:
        if not apply_spectra and subdir in ('Spectra', 'Spectralite'):
            print(f'  [SKIP spectra]  {instrument}/{subdir}')
            continue
        src_dir = src_inst / subdir if subdir else src_inst
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.glob('*.parquet')):
            if wyo_only and is_mml(path):
                n_skip += 1
                continue
            rs       = raw_stem(path)
            dst_base = dst_inst / subdir if subdir else dst_inst
            if rs in rejected_stems:
                dst_path = dst_base / 'bad' / path.name
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst_path)
                print(f'  [BAD]  {path.name:<55}  -> bad/')
                n_bad += 1
                continue
            lag_s = lags.get(rs)
            if lag_s is None:
                print(f'  [WARN no lag]  {instrument}/{subdir or ""}/{path.name} — using 0s')
                lag_s = 0.0; n_warn += 1
            dst_path = dst_base / path.name
            rows = apply_lag_to_parquet(path, lag_s, dst_path, ts_status='picarro_aligned')
            print(f'  [OK]  {path.name:<55}  {lag_s:>+6.0f}s  [{rows:,} rows]')
            n_ok += 1
    print(f'  -> {instrument}: aligned={n_ok}, bad={n_bad}, warn={n_warn}, skipped_mml={n_skip}')
    return {'ok': n_ok, 'bad': n_bad, 'warn': n_warn, 'skipped_mml': n_skip}

print('Apply helpers loaded.')

In [ ]:
APPLY_SPECTRA = True

INSTRUMENT_SUBDIRS_WYO = {
    'WYO_aerisultra460':  (['Raw', 'Eng', 'Spectralite'], False),
    'LANL_aerisultra321': (['Raw', 'Eng', 'Spectra'],     True),
    'LANL_aerispico017':  (['Raw', 'Eng', 'Spectra'],     True),
}

with open(STAGE_03_DIR / 'lag_offsets_wyo.json') as fh:
    saved = json.load(fh)

apply_stats = {}
for inst, (subdirs, wyo_only) in INSTRUMENT_SUBDIRS_WYO.items():
    label = '(WYO dates only)' if wyo_only else ''
    print(f'\n{"="*60}\n  {inst}  {label}\n{"="*60}')
    lags     = saved['lags'].get(inst, {})
    rejected = set(saved['rejected'].get(inst, []))
    stats    = apply_instrument(inst, subdirs, lags, rejected,
                                apply_spectra=APPLY_SPECTRA, wyo_only=wyo_only)
    apply_stats[inst] = stats

apply_manifest = {
    'stage':         '03a_apply_wyo',
    'run_utc':       datetime.now(timezone.utc).isoformat(),
    'git_hash':      saved['git_hash'],
    'git_dirty':     saved['git_dirty'],
    'apply_spectra': APPLY_SPECTRA,
    'instruments':   apply_stats,
}
apply_path = STAGE_03_DIR / 'apply_manifest_wyo.json'
with open(apply_path, 'w') as fh:
    json.dump(apply_manifest, fh, indent=2)

print(f'\nWYO apply complete -> {apply_path}')
print('Run no_coverage and trusted pass-through cells below to finish Stage 03a.')

---
## Pass-through: no_coverage → bad_timestamp

Stage 02 `no_coverage/` files have Mountain Time clocks — nothing Stage 03 can do.
Copied to `bad_timestamp/` so Stage 03 is a complete dataset for analysis.

In [ ]:
NO_COVERAGE_SUBDIRS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
}

passthrough_stats = {}
for inst, subdirs in NO_COVERAGE_SUBDIRS.items():
    n_ok = 0
    for subdir in subdirs:
        if not APPLY_SPECTRA and subdir == 'Spectra':
            continue
        src_dir = STAGE_02_DIR / inst / subdir / 'no_coverage'
        dst_dir = STAGE_03_DIR / inst / subdir / 'bad_timestamp'
        if not src_dir.exists():
            continue
        files = sorted(src_dir.glob('*.parquet'))
        if not files:
            continue
        dst_dir.mkdir(parents=True, exist_ok=True)
        for path in files:
            shutil.copy2(path, dst_dir / path.name)
            print(f'  [PASS]  {inst}/{subdir}/bad_timestamp/{path.name}')
            n_ok += 1
    passthrough_stats[inst] = {'copied': n_ok}
    print(f'  -> {inst}: {n_ok} files -> bad_timestamp/')

apply_path = STAGE_03_DIR / 'apply_manifest_wyo.json'
with open(apply_path) as fh:
    m = json.load(fh)
m['passthrough'] = passthrough_stats
with open(apply_path, 'w') as fh:
    json.dump(m, fh, indent=2)
print(f'\nno_coverage pass-through complete. Manifest updated -> {apply_path}')

---
## Pass-through: trusted instruments

Picarro and Sprinter carry trusted UTC timestamps. Copied from Stage 02 unchanged
so Stage 03 is a complete, self-contained aligned dataset.

In [ ]:
TRUSTED_INSTRUMENTS = ['WYO_picarro', 'WYO_sprinter']

trusted_stats = {}
for inst in TRUSTED_INSTRUMENTS:
    src_dir = STAGE_02_DIR / inst
    dst_dir = STAGE_03_DIR / inst
    files   = sorted(src_dir.glob('*.parquet'))
    if not files:
        print(f'[WARN]  {inst} — no parquet files in {src_dir}')
        trusted_stats[inst] = {'ok': 0}
        continue
    print(f'\n{"="*60}\n  {inst}  ({len(files)} files)\n{"="*60}')
    n_ok = 0
    for f in files:
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst_dir / f.name)
        print(f'  OK  {f.name}')
        n_ok += 1
    trusted_stats[inst] = {'ok': n_ok}

apply_path = STAGE_03_DIR / 'apply_manifest_wyo.json'
with open(apply_path) as fh:
    m = json.load(fh)
m['trusted'] = trusted_stats
with open(apply_path, 'w') as fh:
    json.dump(m, fh, indent=2)
print(f'\nTrusted pass-through complete.')
print(f'Stage 03a complete -> {STAGE_03_DIR}')